In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import when, col

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    VectorAssembler
)
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.functions import floor

In [4]:
spark = SparkSession.builder \
    .appName("FlightDelayML") \
    .getOrCreate()

In [7]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("../data/sample.csv")

In [12]:
# Create target variable (is_delayed = 1 if arrival delay is greater than 15 minutes, else 0)
df = df.withColumn(
    "is_delayed",
    when(col("ARR_DELAY") > 15, 1).otherwise(0)
)

# Extract departure hour
df = df.withColumn(
    "departure_hour",
    floor(col("CRS_DEP_TIME") / 100)
)

# Features
df_features = df.select(
    "MONTH",
    "DAY_OF_MONTH",
    "DAY_OF_WEEK",
    "departure_hour",
    "DISTANCE",
    "OP_UNIQUE_CARRIER",
    "ORIGIN",
    "DEST",
    "is_delayed"
)

# Encoding categorical features
carrier_indexer = StringIndexer(
    inputCol="OP_UNIQUE_CARRIER",
    outputCol="carrier_idx",
    handleInvalid="keep"
)

origin_indexer = StringIndexer(
    inputCol="ORIGIN",
    outputCol="origin_idx",
    handleInvalid="keep"
)

dest_indexer = StringIndexer(
    inputCol="DEST",
    outputCol="dest_idx",
    handleInvalid="keep"
)

# Assemble feature vector
assembler = VectorAssembler(
    inputCols=[
        "MONTH",
        "DAY_OF_MONTH",
        "DAY_OF_WEEK",
        "departure_hour",
        "DISTANCE",
        "carrier_idx",
        "origin_idx",
        "dest_idx"
    ],
    outputCol="features"
)

### Model development

In [17]:
lr = LogisticRegression(
    labelCol="is_delayed",
    featuresCol="features",
    maxIter=10,
    regParam=0.01
)

# Build Pipeline
pipeline = Pipeline(stages=[
    carrier_indexer,
    origin_indexer,
    dest_indexer,
    assembler,
    lr
])

# Train/Test Split
train_df, test_df = df_features.randomSplit(
    [0.8, 0.2],
    seed=42
)

print(f"Training Records: {train_df.count()}")
print(f"Testing Records: {test_df.count()}")

Training Records: 2051
Testing Records: 449


### Model training / predictions

In [21]:
# Train Model
model = pipeline.fit(train_df)

# Generate Predictions
predictions = model.transform(test_df)

print("\nSample Predictions:")
predictions.select(
    "is_delayed",
    "prediction",
    "probability"
).show(10, truncate=False)


Sample Predictions:
+----------+----------+----------------------------------------+
|is_delayed|prediction|probability                             |
+----------+----------+----------------------------------------+
|1         |0.0       |[0.8533650874402372,0.1466349125597628] |
|0         |0.0       |[0.8595488421802094,0.14045115781979056]|
|0         |0.0       |[0.91687769635794,0.08312230364205997]  |
|0         |0.0       |[0.8743310771237963,0.12566892287620368]|
|0         |0.0       |[0.8849510486793696,0.11504895132063042]|
|0         |0.0       |[0.8713852531791547,0.12861474682084528]|
|1         |0.0       |[0.8578205728222345,0.14217942717776555]|
|0         |0.0       |[0.8396743605090075,0.16032563949099254]|
|0         |0.0       |[0.890639746270209,0.10936025372979097] |
|0         |0.0       |[0.8693958841720866,0.13060411582791343]|
+----------+----------+----------------------------------------+
only showing top 10 rows


### Model Evaluation

In [22]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy_eval = MulticlassClassificationEvaluator(
    labelCol="is_delayed",
    predictionCol="prediction",
    metricName="accuracy"
)

precision_eval = MulticlassClassificationEvaluator(
    labelCol="is_delayed",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_eval = MulticlassClassificationEvaluator(
    labelCol="is_delayed",
    predictionCol="prediction",
    metricName="weightedRecall"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol="is_delayed",
    predictionCol="prediction",
    metricName="f1"
)

print("Accuracy:", accuracy_eval.evaluate(predictions))
print("Precision:", precision_eval.evaluate(predictions))
print("Recall:", recall_eval.evaluate(predictions))
print("F1:", f1_eval.evaluate(predictions))

Accuracy: 0.8017817371937639
Precision: 0.6428539540974499
Recall: 0.8017817371937639
F1: 0.7135758353269592


In [26]:
# Save Model
model.write().overwrite().save("../models/flight_delay_model")